# PCL detection (RoBERTa)

RoBERTa-base, class-weighted BCE, auxiliary loss on the 7 PCL categories, threshold tuning on dev. Two-stage: train on train set, pick best epoch and threshold, then retrain on train+dev and predict.

In [10]:
import os, ast, random, warnings, logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [11]:
CFG = {
    'model_name': 'roberta-base',
    'main_data': 'data/dontpatronizeme_pcl.tsv',
    'train_ids': 'data/train_semeval_parids-labels.csv',
    'dev_ids': 'data/dev_semeval_parids-labels.csv',
    'test_data': 'data/task4_test.tsv',
    'max_length': 256,
    'batch_size': 16,
    'grad_accum': 2,
    'lr': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'num_epochs': 8,
    'patience': 3,
    'aux_weight': 0.3,
    'seed': 42,
    'best_model': 'BestModel/best_model_roberta.pt',
    'final_model': 'BestModel/final_model_roberta.pt',
    'dev_preds': 'BestModel/dev.txt',
    'test_preds': 'BestModel/test.txt',
}

PCL_CATEGORIES = [
    'Unbalanced power relations', 'Shallow solution', 'Presupposition',
    'Authority voice', 'Metaphor', 'Compassion', 'The poorer the merrier',
]

os.makedirs('BestModel', exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(CFG['seed'])

In [12]:
def load_main_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) == 6 and parts[0].strip().isdigit():
                rows.append({'par_id': int(parts[0]), 'keyword': parts[2], 'country': parts[3],
                             'text': parts[4].strip(), 'label_raw': int(parts[5])})
    df = pd.DataFrame(rows)
    df['label_binary'] = (df['label_raw'] >= 2).astype(int)
    return df

def load_split_ids(path):
    df = pd.read_csv(path)
    df['par_id'] = df['par_id'].astype(int)
    df['label_list'] = df['label'].apply(ast.literal_eval)
    return df[['par_id', 'label_list']]

def load_test_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) >= 5 and parts[0].strip().startswith('t_'):
                rows.append({'par_id': parts[0].strip(), 'text': parts[4].strip() if len(parts) > 4 else ''})
    df = pd.DataFrame(rows)
    return df[df['text'].str.len() > 0].reset_index(drop=True)

df_main = load_main_data(CFG['main_data'])
df_train_ids = load_split_ids(CFG['train_ids'])
df_dev_ids = load_split_ids(CFG['dev_ids'])
df_test = load_test_data(CFG['test_data'])

df_train = df_main[df_main['par_id'].isin(df_train_ids['par_id'])].copy()
df_dev = df_main[df_main['par_id'].isin(df_dev_ids['par_id'])].copy()

df_train = df_train.merge(df_train_ids, on='par_id', how='left').reset_index(drop=True)
df_dev = df_dev.merge(df_dev_ids, on='par_id', how='left').reset_index(drop=True)

def to_aux_list(x):
    return x if isinstance(x, list) else [0] * 7
df_train['label_list'] = df_train['label_list'].apply(to_aux_list)
df_dev['label_list'] = df_dev['label_list'].apply(to_aux_list)

n_pos = (df_train['label_binary'] == 1).sum()
n_neg = (df_train['label_binary'] == 0).sum()
print(f"Train: {len(df_train)}, Dev: {len(df_dev)}, Test: {len(df_test)}")
print(f"Train PCL: {n_pos}, No-PCL: {n_neg}  ratio {n_neg/n_pos:.2f}:1")

Train: 8375, Dev: 2094, Test: 3832
Train PCL: 794, No-PCL: 7581  ratio 9.55:1


In [13]:
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length, aux_labels=None):
        self.texts = [str(t) for t in texts]
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.aux_labels = aux_labels

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length,
                             padding='max_length', truncation=True, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(float(self.labels[idx]), dtype=torch.float)
        if self.aux_labels is not None:
            item['aux_labels'] = torch.tensor(self.aux_labels[idx], dtype=torch.float)
        return item

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

train_ds = PCLDataset(df_train['text'].tolist(), df_train['label_binary'].tolist(),
    tokenizer, CFG['max_length'], aux_labels=df_train['label_list'].tolist())
dev_ds = PCLDataset(df_dev['text'].tolist(), df_dev['label_binary'].tolist(),
    tokenizer, CFG['max_length'], aux_labels=df_dev['label_list'].tolist())
test_ds = PCLDataset(df_test['text'].tolist(), [0]*len(df_test), tokenizer, CFG['max_length'])

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
dev_dl = DataLoader(dev_ds, batch_size=CFG['batch_size']*2, shuffle=False, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=CFG['batch_size']*2, shuffle=False, num_workers=0)

pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"pos_weight = {pos_weight.item():.2f}")

train_arr = np.array(df_train['label_list'].tolist(), dtype=np.float32)
pos_counts = train_arr.sum(axis=0).clip(min=1)
neg_counts = len(train_arr) - pos_counts
pw_aux = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(device)
aux_criterion = nn.BCEWithLogitsLoss(pos_weight=pw_aux)
for cat, w in zip(PCL_CATEGORIES, pw_aux.cpu().numpy()):
    print(cat, f"{w:.1f}")

pos_weight = 9.55
Unbalanced power relations 13.6
Shallow solution 51.3
Presupposition 50.7
Authority voice 42.6
Metaphor 56.8
Compassion 22.1
The poorer the merrier 287.8


In [14]:
def run_epoch(model, loader, optimizer, scheduler, criterion, grad_accum,
              aux_criterion=None, aux_weight=0.3):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='Train', leave=False)):
        logits = model(batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device)).logits
        loss = criterion(logits[:, 0], batch['labels'].to(device))
        if aux_criterion is not None and 'aux_labels' in batch:
            loss = loss + aux_weight * aux_criterion(logits[:, 1:], batch['aux_labels'].to(device))
        loss = loss / grad_accum
        loss.backward()
        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * grad_accum
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        logits = model(batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device)).logits
        all_logits.append(logits[:, 0].cpu().numpy())
        all_labels.append(batch['labels'].numpy())
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)
    return f1_score(all_labels.astype(int), preds, zero_division=0), probs, all_labels

def tune_threshold(probs, labels, t_min=0.30, t_max=0.70):
    best_t, best_f = 0.5, 0.0
    for t in np.linspace(t_min, t_max, 41):
        f = f1_score(labels.astype(int), (probs >= t).astype(int), zero_division=0)
        if f > best_f: best_f, best_t = f, t
    print(f"Best threshold: {best_t:.3f}  →  dev F1 = {best_f:.4f}")
    return float(best_t)

@torch.no_grad()
def predict(model, loader, threshold):
    model.eval()
    probs = []
    for batch in tqdm(loader, desc='Predict', leave=False):
        logits = model(batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device)).logits
        probs.append(torch.sigmoid(logits[:, 0]).cpu().numpy())
    probs = np.concatenate(probs)
    return (probs >= threshold).astype(int), probs

In [15]:
set_seed(CFG['seed'])
model = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], num_labels=8).to(device)
total_steps = (len(train_dl) // CFG['grad_accum']) * CFG['num_epochs']
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*CFG['warmup_ratio']), total_steps)
print(f"Model: {CFG['model_name']}  |  num_labels=8  |  total steps: {total_steps}")

best_f1, best_epoch, best_probs, best_labels = 0.0, 0, None, None
patience_cnt = 0

for epoch in range(CFG['num_epochs']):
    print(f"\nEpoch {epoch+1}/{CFG['num_epochs']}")
    train_loss = run_epoch(model, train_dl, optimizer, scheduler, criterion, CFG['grad_accum'],
                           aux_criterion=aux_criterion, aux_weight=CFG['aux_weight'])
    f1, probs, labels = evaluate(model, dev_dl, threshold=0.5)
    print(f"   Train loss: {train_loss:.4f}  |  Dev F1 (t=0.5): {f1:.4f}", end='')
    if f1 > best_f1:
        best_f1, best_epoch, best_probs, best_labels = f1, epoch+1, probs.copy(), labels.copy()
        patience_cnt = 0
        torch.save(model.state_dict(), CFG['best_model'])
        print(" best")
    else:
        patience_cnt += 1
        print(f" patience {patience_cnt}/{CFG['patience']}")
        if patience_cnt >= CFG['patience']:
            break

print(f"\nBest dev F1 = {best_f1:.4f} at epoch {best_epoch}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1821.33it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


Model: roberta-base  |  num_labels=8  |  total steps: 2096

Epoch 1/8


   Train loss: 1.4264  |  Dev F1 (t=0.5): 0.4890 best

Epoch 2/8


   Train loss: 1.0303  |  Dev F1 (t=0.5): 0.5412 best

Epoch 3/8


   Train loss: 0.8002  |  Dev F1 (t=0.5): 0.5769 best

Epoch 4/8


   Train loss: 0.6186  |  Dev F1 (t=0.5): 0.5817 best

Epoch 5/8


   Train loss: 0.4601  |  Dev F1 (t=0.5): 0.5747 patience 1/3

Epoch 6/8


   Train loss: 0.2904  |  Dev F1 (t=0.5): 0.6114 best

Epoch 7/8


   Train loss: 0.1951  |  Dev F1 (t=0.5): 0.6072 patience 1/3

Epoch 8/8


   Train loss: 0.1650  |  Dev F1 (t=0.5): 0.6085 patience 2/3

Best dev F1 = 0.6114 at epoch 6


In [16]:
model.load_state_dict(torch.load(CFG['best_model'], map_location=device))
_, best_probs, best_labels = evaluate(model, dev_dl, threshold=0.5)
best_thresh = tune_threshold(best_probs, best_labels)
preds_dev = (best_probs >= best_thresh).astype(int)
print(classification_report(best_labels.astype(int), preds_dev, target_names=['No-PCL', 'PCL'], digits=4))
with open(CFG['dev_preds'], 'w') as f:
    for p in preds_dev: f.write(f"{int(p)}\n")
print(f"Saved {CFG['dev_preds']}")

Best threshold: 0.690  →  dev F1 = 0.6150
              precision    recall  f1-score   support

      No-PCL     0.9617    0.9541    0.9579      1895
         PCL     0.5935    0.6382    0.6150       199

    accuracy                         0.9241      2094
   macro avg     0.7776    0.7961    0.7864      2094
weighted avg     0.9267    0.9241    0.9253      2094

Saved BestModel/dev.txt


In [17]:
df_full = pd.concat([df_train, df_dev], ignore_index=True)
n_pos_f = (df_full['label_binary'] == 1).sum()
n_neg_f = (df_full['label_binary'] == 0).sum()

full_ds = PCLDataset(df_full['text'].tolist(), df_full['label_binary'].tolist(),
    tokenizer, CFG['max_length'], aux_labels=df_full['label_list'].tolist())
full_dl = DataLoader(full_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=0, pin_memory=True)

pos_weight_f = torch.tensor([n_neg_f / n_pos_f], dtype=torch.float).to(device)
criterion_f = nn.BCEWithLogitsLoss(pos_weight=pos_weight_f)

full_arr = np.array(df_full['label_list'].tolist(), dtype=np.float32)
pw_aux_f = torch.tensor((len(full_arr) - full_arr.sum(0)) / full_arr.sum(0).clip(1), dtype=torch.float).to(device)
aux_criterion_f = nn.BCEWithLogitsLoss(pos_weight=pw_aux_f)

set_seed(CFG['seed'])
model_final = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], num_labels=8).to(device)
steps_s2 = (len(full_dl) // CFG['grad_accum']) * best_epoch
opt_s2 = AdamW(model_final.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_s2 = get_linear_schedule_with_warmup(opt_s2, int(steps_s2*CFG['warmup_ratio']), steps_s2)
print(f"Retrain {best_epoch} epochs on train+dev ({len(df_full)} samples)")

for epoch in range(best_epoch):
    print(f"epoch {epoch+1}/{best_epoch}")
    run_epoch(model_final, full_dl, opt_s2, sched_s2, criterion_f, CFG['grad_accum'],
              aux_criterion=aux_criterion_f, aux_weight=CFG['aux_weight'])

torch.save(model_final.state_dict(), CFG['final_model'])

preds_test, probs_test = predict(model_final, test_dl, threshold=best_thresh)
np.save('BestModel/test_probs.npy', probs_test)

with open(CFG['test_preds'], 'w') as f:
    for p in preds_test: f.write(f"{int(p)}\n")
print(f"Saved {CFG['test_preds']} (threshold={best_thresh:.3f})")
print(f"PCL preds: {preds_test.sum()}/{len(preds_test)}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1768.72it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


Retrain 6 epochs on train+dev (10469 samples)
epoch 1/6


epoch 2/6


epoch 3/6


epoch 4/6


epoch 5/6


epoch 6/6


Saved BestModel/test.txt (threshold=0.690)
PCL preds: 285/3832
